SENTENCE TRANSFORMERS


In [81]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer , util
import numpy as np


In [6]:
print("="*60)
print("Loading the Tranformer Model")
print("="* 60)

Loading the Tranformer Model


In [7]:
model = SentenceTransformer("all-MiniLM-L6-V2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
print(f"Model Loaded : {model.get_sentence_embedding_dimension()}")

Model Loaded : 384


In [10]:
print("="*60)
print("What Does an Embedding look like:")
print("="* 60)

What Does an Embedding look like:


In [11]:
sample = "Attention Mechanisms allows machine learning models or tranformers to weigh relevance between sentences"

In [12]:
embedding = model.encode(sample , convert_to_tensor= True)

In [18]:
print(f"Input Text : {sample}")
print(f"Output type : {embedding.type()}")
print(f"Output Shape : {embedding.shape}")
print(f"First 8 values : {embedding[:8].tolist()}")

Input Text : Attention Mechanisms allows machine learning models or tranformers to weigh relevance between sentences
Output type : torch.FloatTensor
Output Shape : torch.Size([384])
First 8 values : [0.03671996667981148, 0.016192443668842316, 0.0005673713749274611, 0.03661426529288292, 0.08493495732545853, 0.05494362860918045, 0.04962673410773277, 0.06593618541955948]


In [19]:
print("="*60)
print("Co-sine Similarity")
print("="* 60)

Co-sine Similarity


In [33]:
def co_sinSim(a:torch.tensor , b : torch.tensor)-> float :
    return F.cosine_similarity(a.unsqueeze(0) , b.unsqueeze(0)).item()

In [22]:
s1 = "Transformers use self-attention to capture long-range dependencies."
s2 = "Self-attention lets the model relate distant words in a sequence."
s3 = "Stochastic gradient descent minimises the training loss."   

In [24]:
e1 , e2 , e3 = model.encode([s1,s2,s3],convert_to_tensor=True)

In [39]:
sim_1_2 = co_sinSim(e1 , e2)
sim_1_3 = co_sinSim(e1 , e3)
sim_2_3 = co_sinSim(e2 , e3)

In [40]:
print(f"S1 :{s1}")
print(f"S2 :{s2}")
print(f"S3 :{s3}")


S1 :Transformers use self-attention to capture long-range dependencies.
S2 :Self-attention lets the model relate distant words in a sequence.
S3 :Stochastic gradient descent minimises the training loss.


In [42]:
print(f"sim(S1,S2)={sim_1_2:.4f}")
print(f"sim(S1,S3)={sim_1_3:.4f}")
print(f"sim(S2,S3)={sim_2_3:.4f}")

sim(S1,S2)=0.4664
sim(S1,S3)=0.1409
sim(S2,S3)=0.2547


In [43]:
print("="*60)
print("Mini Rag System")
print("="* 60)

Mini Rag System


In [44]:
paper_chunks = [
    
    "The transformer architecture relies on multi-head self-attention to model "
    "relationships between all pairs of tokens in a sequence simultaneously.",
 
    # chunk 1
    "Batch normalisation stabilises training by normalising layer inputs, "
    "reducing internal covariate shift and allowing higher learning rates.",
 
    # chunk 2
    "BERT pre-trains a deep bidirectional transformer by jointly conditioning "
    "on both left and right context using a masked language modelling objective.",
 
    # chunk 3
    "Dropout randomly zeroes activations during training, acting as an implicit "
    "ensemble method that reduces overfitting in deep neural networks.",
 
    # chunk 4
    "The attention score is computed as the scaled dot-product of query and key "
    "vectors: softmax(QK^T / sqrt(d_k)) * V, where d_k is the key dimension."
]

In [46]:
chunk_embeddings = model.encode(paper_chunks,convert_to_tensor=True)
print(f"Chunk Embedding matrix shape: {chunk_embeddings.shape}")

Chunk Embedding matrix shape: torch.Size([5, 384])


In [50]:
user_query = "How does the attention mechanism compute its output?"

In [48]:
query_embedding = model.encode(user_query,convert_to_tensor=True)

In [53]:
scores = []

for idx , chunk_emb in enumerate(chunk_embeddings):
    score = co_sinSim(query_embedding,chunk_emb)
    scores.append((score , idx))

scores.sort(reverse=True)

In [63]:
print(f"Retrieval Results(ranked by cosine Similarity)")
for rank , (score , idx) in enumerate(scores, start = 1):
    preview = paper_chunks[idx][:20]+"..."
    print(f"Rank : {rank} | score : {score:.4f} |  chunk :  {idx} |  preview :  {preview} | ") 

Retrieval Results(ranked by cosine Similarity)
Rank : 1 | score : 0.6337 |  chunk :  4 |  preview :  The attention score ... | 
Rank : 2 | score : 0.4322 |  chunk :  0 |  preview :  The transformer arch... | 
Rank : 3 | score : 0.2798 |  chunk :  2 |  preview :  BERT pre-trains a de... | 
Rank : 4 | score : 0.2343 |  chunk :  1 |  preview :  Batch normalisation ... | 
Rank : 5 | score : 0.1987 |  chunk :  3 |  preview :  Dropout randomly zer... | 


In [65]:
top_chunk_idx = scores[0][1]

print(f"TOP CHUNK IS : {top_chunk_idx}")

TOP CHUNK IS : 4


In [66]:
print("="*60)
print("Batch Encoding and Normalization")
print("="* 60)

Batch Encoding and Normalization


In [67]:
texts = [
    "Neural networks learn hierarchical feature representations.",
    "Recurrent networks process sequences step by step.",
    "Convolutional networks exploit spatial locality in images.",
]

In [68]:
batch_embs = model.encode(texts , convert_to_tensor=True , normalize_embeddings=True)

In [70]:
print(f"Batch Input:{len(texts)} Sentences")
print(f"Output Shape : {batch_embs.shape}")

Batch Input:3 Sentences
Output Shape : torch.Size([3, 384])


In [71]:
norms = torch.linalg.norm(batch_embs,dim=1)
print(f"L2 Norms after normalization : {norms.tolist()}")

L2 Norms after normalization : [1.0, 1.0, 1.0]


In [73]:
dot_product_sim = (batch_embs[0] @ batch_embs[1]).item()
print(f"\ndot product sim(sent0 , sent1) = {dot_product_sim:.4f}")
print(f"Cosine sim(sent0, sent1) = {co_sinSim(batch_embs[0],batch_embs[1]):.4f}")



dot product sim(sent0 , sent1) = 0.4089
Cosine sim(sent0, sent1) = 0.4089


EXERCISES:



In [74]:
print("Embedding 10 Differents Sentences and printing a similarity matrtix")

Embedding 10 Differents Sentences and printing a similarity matrtix


In [76]:
samples = [
    "The old lighthouse overlooked a quiet bay where dolphins appeared every sunrise.",
    "A programmer accidentally trained a model to recognize different kinds of sandwiches.",
    "Heavy rain forced the hikers to spend the night inside an abandoned wooden cabin.",
    "My neighbor collects vintage clocks but never bothers to set the correct time.",
    "Scientists discovered that certain plants communicate through underground fungal networks.",
    "The chef added cinnamon to the soup, creating an unexpectedly delicious flavor.",
    "A bright red bicycle was parked outside the library despite the snowy weather.",
    "Children built an enormous cardboard castle that survived the entire weekend.",
    "The spacecraft transmitted mysterious signals just before entering the asteroid belt.",
    "Every Friday evening, musicians gather in the town square to perform for anyone passing by."
]

In [92]:
samples_embed = model.encode(samples , convert_to_tensor=True)
print(f"Sample Embeddings shape : {samples_embed.shape}")


Sample Embeddings shape : torch.Size([10, 384])


In [85]:
matrix = util.cos_sim(samples_embed,samples_embed)

In [89]:
from tabulate import tabulate
print(tabulate(matrix.tolist(),tablefmt="double_grid"))

╔════════════╦════════════╦════════════╦══════════════╦═════════════╦═════════════╦════════════╦════════════╦════════════╦═════════════╗
║  1         ║ -0.0483903 ║  0.12774   ║  0.128779    ║ 0.0129786   ║  0.0645654  ║  0.0676884 ║  0.137411  ║  0.0719043 ║  0.0977421  ║
╠════════════╬════════════╬════════════╬══════════════╬═════════════╬═════════════╬════════════╬════════════╬════════════╬═════════════╣
║ -0.0483903 ║  1         ║ -0.0385823 ║  0.115563    ║ 0.112217    ║  0.272793   ║  0.0226741 ║  0.0195136 ║  0.201027  ║ -0.0242306  ║
╠════════════╬════════════╬════════════╬══════════════╬═════════════╬═════════════╬════════════╬════════════╬════════════╬═════════════╣
║  0.12774   ║ -0.0385823 ║  1         ║  0.0315899   ║ 0.112061    ║  0.0275344  ║  0.16136   ║  0.244944  ║  0.0243104 ║  0.104697   ║
╠════════════╬════════════╬════════════╬══════════════╬═════════════╬═════════════╬════════════╬════════════╬════════════╬═════════════╣
║  0.128779  ║  0.115563  ║  0.0315899 ║ 

In [94]:
matrix.max()

tensor(1.0000)